# Подготовка датасета



In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random



In [1]:
# Импорт библиотек
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Фиксируем seed для воспроизводимости
np.random.seed(42)

# Параметры датасета
n_rows = 120_000

# Генерация данных
data = {
    # Уникальный строковый идентификатор заказа
    'order_id': [f"ORD-{i:07d}" for i in range(1, n_rows + 1)],

    # Дата и время заказа (случайный день и час в 2024 году)
    'order_date': [
        (datetime(2024, 1, 1) + timedelta(days=int(d), hours=int(h)))
        for d, h in zip(
            np.random.randint(0, 365, n_rows),
            np.random.randint(0, 24, n_rows)
        )
    ],

    # Категория товара — главный категориальный признак для groupBy
    'product_category': np.random.choice(
        ['Electronics', 'Clothing', 'Home_Goods', 'Books', 'Sports'],
        size=n_rows,
        p=[0.20, 0.25, 0.20, 0.15, 0.20]
    ),

    # Количество товаров в заказе
    'quantity': np.random.randint(1, 15, size=n_rows),

    # Цена за единицу: логнормальное распределение имитирует реальные цены
    'unit_price': np.round(
        np.random.lognormal(mean=3.2, sigma=0.9, size=n_rows), 2
    ),

    # Регион доставки — второй категориальный признак
    'delivery_region': np.random.choice(
        ['Moscow', 'SPb', 'Kazan', 'Novosibirsk', 'Remote'],
        size=n_rows,
        p=[0.30, 0.20, 0.15, 0.15, 0.20]
    ),

    # Флаг экспресс-доставки (25% заказов — экспресс)
    'is_express': np.random.choice([True, False], size=n_rows, p=[0.25, 0.75]),

    # Оценка клиента от 1 до 5, нормальное распределение с обрезкой по краям
    'customer_rating': np.round(
        np.clip(np.random.normal(loc=4.1, scale=1.0, size=n_rows), 1.0, 5.0), 1
    )
}

df = pd.DataFrame(data)

# Вычисляемый признак: итоговая сумма = количество × цена
df['total_amount'] = df['quantity'] * df['unit_price']

# Переводим дату в строку — Spark и CSV читают этот формат корректно
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Добавляем ~5% пропусков в customer_rating для демонстрации обработки NaN в Spark
mask_null = np.random.random(n_rows) < 0.05
df.loc[mask_null, 'customer_rating'] = np.nan

# Проверка результата
print(f"Размер: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
print(f"\nТипы данных:\n{df.dtypes}")
print(f"\nПропуски:\n{df.isnull().sum()}")
print(f"\nПервые 3 строки:")
df.head(3)

# Сохранение в CSV
df.to_csv('ecommerce_orders.csv', index=False)
print(f"\nФайл сохранён: ecommerce_orders.csv")

Размер: 120,000 строк × 9 столбцов

Типы данных:
order_id                str
order_date              str
product_category        str
quantity              int32
unit_price          float64
delivery_region         str
is_express             bool
customer_rating     float64
total_amount        float64
dtype: object

Пропуски:
order_id               0
order_date             0
product_category       0
quantity               0
unit_price             0
delivery_region        0
is_express             0
customer_rating     6068
total_amount           0
dtype: int64

Первые 3 строки:

Файл сохранён: ecommerce_orders.csv


**Тема датасета**

Имитация записей заказов интернет-магазина за один год.
Каждая строка  — это одна позиция в заказе.

**Столбцы**
* order_id — уникальный номер заказа, обычный ID.
* order_date — дата и время заказа. Хранится как строка в формате ГГГГ-ММ-ДД ЧЧ:ММ:СС.
* product_category — категория товара (электроника, одежда, товары для дома, книги, спорт). Это категориальный признак.
* quantity — количество товаров в заказе.
* unit_price — цена за единицу товара.
* delivery_region — регион доставки (Москва, СПб, Казань, Новосибирск, удаленные регионы). Второй категориальный признак.
* is_express — флаг экспресс-доставки (да/нет).
* customer_rating — оценка клиента от 1 до 5. В этом признаке мы намеренно оставили около 5% пропусков.
* total_amount — итоговая сумма строки заказа. Вычисляемый признак (quantity умножить на unit_price).